# Convolution 2.0

## Import libraries

In [36]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pypher
import warnings
from astropy.io import fits
from astropy.modeling import models, fitting
from astropy.utils.exceptions import AstropyWarning
from pathlib import Path
from astropy.nddata import block_reduce 



## Set directories

Determine file paths and obtain lists of galaxy images and PSF files for each survey inside Input/

In [2]:
CWD = Path.cwd()
ROOT = CWD.parents[1]
ASTROVELLO_DIR = ROOT / "AsTrovello_2_0"

input_dir = ASTROVELLO_DIR / "Input"

survey_paths = list(input_dir.glob("*"))
survey_names = [f.name for f in survey_paths]

galaxy = "ngc1097"

survey_names

['PHANGS', 'S4G']

## Obtain all survey files (Science images and PSFs)

In [3]:
image_files = []
psf_files = []
for survey in survey_names:
    image_dir = input_dir / survey / "galaxies" / galaxy
    psf_dir = input_dir / survey / "PSF"

    if survey == "PHANGS":
        current_image_files = list(image_dir.glob("*_exp-drc-sci.fits"))
        current_psf_files = list(psf_dir.glob("*PSFSTD*.fits"))
    elif survey == "S4G":
        current_image_files = list(image_dir.glob(f"{galaxy.upper()}.phot.*.fits"))
        current_psf_files = list(psf_dir.glob("*_col129_row129.fits"))

    image_files = image_files + current_image_files
    psf_files = psf_files + current_psf_files


In [4]:
image_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f555w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f814w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.1.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.2.fits')]

In [5]:
psf_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F275W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F336W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F438W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F555W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F814W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC1_col129_row129.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC2_col129_row129.fits')]

## Determine PSF resolutions 
Calculate FWHM to determine each surveys resolution and define the master file for convolution (lowest resolution).

In [6]:
phangs_img_header = fits.getheader(image_files[2], 0) 
phangs_img_header["INSTRUME"]

'IRAC'

In [7]:
SURVEY_CONFIG = {
                    "PHANGS": 
                    {
                        "TELESCOP": "HST",
                        "INSTRUME": "WFC3",
                        "pixel_scale_arcsec": 0.0395,
                        "binned_factor": 4,
                        "unit_type": "electrons/s", # usado em units.py
                        "force_tan_sip": False
                    },
                    "S4G":
                    {
                        "TELESCOP": "Spitzer",
                        "INSTRUME": "IRAC",
                        "pixel_scale_arcsec": 
                        {
                            1: 1.221, # Channel 1
                            2: 1.223 # Channel 2
                        },
                        "binned_factor": 5,
                        "unit_type": "mjy/sr", # usado em units.py
                        "force_tan_sip": True
                    }
                }

In [8]:
supported_instruments = [survey_data["INSTRUME"] for survey_data in SURVEY_CONFIG.values()]
psf_files[0].name

'PSFSTD_WFC3UV_F275W.fits'

### Get FWHM function

In [ ]:
def get_fwhm(data: np.ndarray) -> float:
    """Estimates the Full Width at Half Maximum (FWHM) using a 2D Gaussian fit.
    
    This method is robust against background noise, negative pixels, and large 
    image bounding boxes. It utilizes the Levenberg-Marquardt least squares 
    algorithm to fit a 2D Gaussian model to the data. To account for slightly 
    elliptical PSFs, the effective sigma is calculated as the geometric mean 
    of the X and Y standard deviations.

    Args:
        data (np.ndarray): A 2D array representing the Point Spread Function (PSF) image.

    Returns:
        float: The estimated FWHM measured in pixels.
    """
    # 1. Remove qualquer NaN que possa quebrar o algoritmo
    data_clean = np.nan_to_num(data, nan=0.0)
    
    # 2. Cria uma malha de coordenadas X e Y do mesmo tamanho da imagem
    y, x = np.mgrid[:data_clean.shape[0], :data_clean.shape[1]]
    
    # 3. Estima os parâmetros iniciais (chutes) para ajudar o algoritmo a convergir mais rápido
    max_val = np.max(data_clean)
    y_center, x_center = np.unravel_index(np.argmax(data_clean), data_clean.shape)
    
    # Cria o modelo inicial da Gaussiana
    g_init = models.Gaussian2D(amplitude=max_val, x_mean=x_center, y_mean=y_center, 
                               x_stddev=2.0, y_stddev=2.0)
    
    # 4. Inicializa o algoritmo de ajuste (Levenberg-Marquardt Mínimos Quadrados)
    fit_g = fitting.LevMarLSQFitter()
    
    # 5. Ajusta o modelo aos dados
    with warnings.catch_warnings():
        # Ignora avisos inofensivos do astropy caso a PSF seja muito ruidosa
        warnings.simplefilter('ignore')
        g_fit = fit_g(g_init, x, y, data_clean)
    
    # 6. Extrai o desvio padrão (sigma) do eixo X e Y e tira a média geométrica
    # A média geométrica lida melhor com PSFs ligeiramente elípticas
    sigma_eff = np.sqrt(abs(g_fit.x_stddev.value * g_fit.y_stddev.value))
    
    # 7. Converte Sigma para FWHM em pixels
    fwhm_pixels = 2.3548 * sigma_eff
    
    return float(fwhm_pixels)

### Calculate FWHM function (for file list)

In [ ]:
def calculateFWHM(psf_file_list: list, SURVEY_CONFIG: dict) -> tuple:
    """AsTrovello 2.0
    
    Iterates through a folder of PSF files, filters them by survey,
    and returns dictionaries containing their physical FWHM (in arcsec).

    Determines to which survey the file belongs based on its parent directory
    and extracts the filter name. Then, determines the PSF's binning factor 
    and its correct pixel scale from the SURVEY_CONFIG dictionary. Finally, 
    calculates the FWHM for each filter and returns a FWHM dictionary and a 
    list with valid file names. 

    Args:
        psf_file_list (list): List of PSF files paths.
        SURVEY_CONFIG (dict): Survey configurations dictionary.

    Returns:
        tuple: A tuple containing:
            - FWHM_dict (dict): Dictionary with filter names as keys and FWHM (in arcsec) as values.
            - valid_files (list): List of valid file names (where FWHM was successfully calculated).

    Note:
        For 3D PSF files, the FWHM is calculated over the mean PSF of the cube.
    """
    FWHM_dict, valid_files = {}, []
    
    # Silencia os avisos chatos de cabeçalho do Astropy
    warnings.simplefilter('ignore', category=AstropyWarning)
    
    for file in psf_file_list:
        if file.name.startswith('.'):
            continue

        if 'S4G' in str(file):
            binned_factor = SURVEY_CONFIG["S4G"]["binned_factor"]
            if file.name == 'IRAC1_col129_row129.fits': 
                filter_name = 'irac1'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][1] / binned_factor
            elif file.name == 'IRAC2_col129_row129.fits': 
                filter_name = 'irac2'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][2] / binned_factor
            else: 
                continue
                
        elif 'PHANGS' in str(file):
            filter_name = file.name.replace('.fits', '').split('_')[-1].lower() 
            binned_factor = SURVEY_CONFIG["PHANGS"]["binned_factor"]
            pixscale = SURVEY_CONFIG["PHANGS"]["pixel_scale_arcsec"] / binned_factor
        else: 
            continue

        try:
            with fits.open(file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
                data = next((h.data for h in hdu if h.data is not None), None)
                
                if data is not None:
                    if data.ndim == 3: 
                        data = np.mean(data, axis=0) 
                    
                    # Usa o ajuste Gaussiano (ou a função robusta) que retorna em pixels
                    fwhm_pixels = get_fwhm(data)
                    
                    # Converte para escala física (arcsec) usando a escala de pixel superamostrada
                    FWHM_dict[filter_name] = np.float32(fwhm_pixels * pixscale)
                    
                    valid_files.append(file.name)
                    print(f"Successfully read: {filter_name} (FWHM: {FWHM_dict[filter_name]:.4f} arcsec)")
                    
        except Exception as e:
            print(f"Processing error {file.name}: {e}")
            
    # Restaura os avisos para o resto do seu código
    warnings.simplefilter('default', category=AstropyWarning)
    
    return FWHM_dict, valid_files
    FWHM_dict, valid_files = {}, []
    
    # Silencia os avisos chatos de cabeçalho do Astropy
    warnings.simplefilter('ignore', category=AstropyWarning)
    
    for file in psf_file_list:
        if file.name.startswith('.'):
            continue

        if 'S4G' in str(file):
            binned_factor = SURVEY_CONFIG["S4G"]["binned_factor"]
            if file.name == 'IRAC1_col129_row129.fits': 
                filter_name = 'irac1'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][1] / binned_factor
            elif file.name == 'IRAC2_col129_row129.fits': 
                filter_name = 'irac2'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][2] / binned_factor
            else: 
                continue
                
        elif 'PHANGS' in str(file):
            filter_name = file.name.replace('.fits', '').split('_')[-1].lower() 
            binned_factor = SURVEY_CONFIG["PHANGS"]["binned_factor"]
            pixscale = SURVEY_CONFIG["PHANGS"]["pixel_scale_arcsec"] / binned_factor
        else: 
            continue

        try:
            with fits.open(file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
                data = next((h.data for h in hdu if h.data is not None), None)
                
                if data is not None:
                    if data.ndim == 3: 
                        data = np.mean(data, axis=0) 
                    
                    # Usa o ajuste Gaussiano (ou a função robusta) que retorna em pixels
                    fwhm_pixels = get_fwhm_simple(data)
                    
                    # Converte para escala física (arcsec) usando a escala de pixel superamostrada
                    FWHM_dict[filter_name] = np.float32(fwhm_pixels * pixscale)
                    
                    valid_files.append(file.name)
                    print(f"Successfully read: {filter_name} (FWHM: {FWHM_dict[filter_name]:.4f} arcsec)")
                    
        except Exception as e:
            print(f"Processing error {file.name}: {e}")
            
    # Restaura os avisos para o resto do seu código
    warnings.simplefilter('default', category=AstropyWarning)
    
    return FWHM_dict, valid_files

### 3. Determine lowest resolution

In [26]:
df_fwhm = pd.DataFrame(list(fwhm_dict.items()), columns=["Filter", "FWHM_arcsec"])
df_fwhm = df_fwhm.sort_values(by="FWHM_arcsec").reset_index(drop=True)

print("\nResolutions Table:\n", df_fwhm)

psf_master_name = df_fwhm.iloc[-1]['Filter']
print(f"\n==> Recommended PSF (master): {psf_master_name}")


Resolutions Table:
   Filter  FWHM_arcsec
0  f275w     0.076603
1  f814w     0.079008
2  f336w     0.081011
3  f555w     0.082506
4  f438w     0.083771
5  irac2     1.525301
6  irac1     1.561972

==> Recommended PSF (master): irac1


## Clean PSFs

In [ ]:
def final_clean_psf(input_file, output_file, SURVEY_CONFIG):
    """
    Standardizes PSF headers and performs true downsampling for PyPHER compatibility.
    Calculates pixel scales, bins down oversampled PSFs, and ensures correct 
    centering and coordinate keywords.
    """
    if 'WFC3UV' in input_file:
        # HST scale: native 0.0395"/pix. We will bin down the 4x oversampled data.
        pixel_scale_arcsec = SURVEY_CONFIG["PHANGS"]["pixel_scale_arcsec"]
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file, ignore_missing_end=True) as hdu:
            # Average the PSF cube to get a 2D representative PSF
            data_2d = np.mean(hdu[0].data, axis=0)
            
            # Downsample the array by a factor of 4 (summing blocks of 4x4 pixels)
            data_2d_binned = block_reduce(data_2d, block_size=4, func=np.sum)
            
            # Force odd parity: PyPHER prefers kernels/PSFs with an odd number of pixels
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            
            # Inject WCS keywords required by PyPHER/Astropy
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 4x): {os.path.basename(output_file)}")

    elif any(x in input_file for x in ['IRAC1', 'IRAC2']):
        # Spitzer scale: native ~1.22"/pix. We will bin down the 5x oversampled data.
        pixel_scale_arcsec = 1.221 if 'IRAC1' in input_file else 1.213
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file) as hdu:
            data_raw = hdu[0].data
            
            # Downsample the array by a factor of 5
            data_2d_binned = block_reduce(data_raw, block_size=5, func=np.sum)
            
            # Force odd parity
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 5x): {os.path.basename(output_file)}")


In [ ]:
def clean_psf(input_file: str, output_file: str, pixel_scale_arcsec: float, binned_factor: int):
    """Standardizes PSF headers and performs true downsampling for PyPHER compatibility.
    
    Agnostic function that calculates pixel scales, bins down oversampled PSFs, 
    forces odd parity, normalizes flux, and ensures correct centering and WCS keywords.

    Args:
        input_file (str): Path to the input PSF FITS file.
        output_file (str): Path to save the cleaned PSF.
        pixel_scale_arcsec (float): Native pixel scale of the instrument.
        binned_factor (int): Factor by which the PSF is oversampled.
        is_3d (bool): If True, averages the data along axis 0 to create a 2D representation.
    """
    pixel_scale_deg = pixel_scale_arcsec / 3600.0

    with fits.open(input_file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
        data = next((h.data for h in hdu if h.data is not None), None)
        
        if data is None:
            print(f"==> Error: No valid data found in {input_file}")
            return

        # 1. Trata cubos 3D (ex: WFC3 do Hubble)
        if data.ndim == 3:
            data = np.mean(data, axis=0)

        # 2. Aplica o downsampling apenas se o fator for maior que 1
        if binned_factor > 1:
            data_processed = block_reduce(data, block_size=binned_factor, func=np.sum)
        else:
            data_processed = data.copy()

        # 3. Força paridade ímpar para o PyPHER
        if data_processed.shape[0] % 2 == 0:
            data_processed = data_processed[:-1, :-1]
            
        # 4. Normaliza para assegurar conservação de fluxo
        data_processed = data_processed / np.sum(data_processed)

        # 5. Criação do novo FITS e injeção do WCS
        new_hdu = fits.PrimaryHDU(data_processed)
        new_hdu.header.update({
            'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
            'CRVAL1': 0.0, 'CRVAL2': 0.0,
            'CRPIX1': (data_processed.shape[1] // 2) + 1, 'CRPIX2': (data_processed.shape[0] // 2) + 1,
            'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
            'PIXSCALE': pixel_scale_arcsec
        })
        
        new_hdu.writeto(output_file, overwrite=True)
        print(f"==> File ready for PyPHER (Binned {binned_factor}x): {os.path.basename(output_file)}")

In [38]:
def identify_psf_survey(filename: Path, SURVEY_CONFIG: dict) -> tuple[float, int]:
    full_filename = str(filename).lower()
    short_filename = filename.name.lower()

    if ("phangs" in full_filename) or ("wfc3" in full_filename):
        survey = "PHANGS"
        pixel_scale_arcsec = SURVEY_CONFIG[survey]["pixel_scale_arcsec"]
        binned_factor = SURVEY_CONFIG[survey]["binned_factor"]
    elif ("s4g" in full_filename) or ("irac" in full_filename):
        survey = "S4G"
        channel = short_filename.split("_")[0].replace("irac", "")
        channel = int(channel)
        pixel_scale_arcsec = SURVEY_CONFIG[survey]["pixel_scale_arcsec"][channel]
        binned_factor = SURVEY_CONFIG[survey]["binned_factor"]
    else:
        raise ValueError(f"Survey unidentified! Please provide available survey for: {filename}")

    return float(pixel_scale_arcsec), int(binned_factor)

4